# BackTrader

## Importing Libraries

In [1]:
import backtrader as bt
import yfinance as yf
import pandas as pd
from datetime import datetime

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 9, 1)

stock_ticker = "AAPL"

In [2]:
data = yf.download(stock_ticker, start=start_date, end=end_date, group_by="ticker")
df = data["AAPL"]
df.head()

C:\Users\shash\AppData\Local\Temp\ipykernel_11128\1577406546.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(stock_ticker, start=start_date, end=end_date, group_by="ticker")
[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Date,,,,,
2025-01-02,248.049428,248.218840,240.964594,242.987411,55740700
2025-01-03,242.499161,243.316252,241.034359,242.499161,40244100
2025-01-06,243.445785,246.455106,242.339711,244.133347,45045600
2025-01-07,242.120491,244.681407,240.496267,241.353226,40856000
2025-01-08,241.064252,242.847929,239.200872,241.841492,37628900


In [ ]:
datafeed = bt.feeds.PandasData(dataname=df)
cerebro = bt.Cerebro()

cerebro.adddata(datafeed)
cerebro.run()
cerebro.plot()

<IPython.core.display.Javascript object>

[[<Figure size 640x480 with 4 Axes>]]

In [5]:
class MyStrategy(bt.Strategy):
    def __init__(self):
        self.close_data = self.data.close

    def next(self):
        print(f"Date: {self.data.datetime.date(0)} Close Price: {self.close_data[0]}")


cerebro.addstrategy(MyStrategy)
cerebro.run()

Close Price: 242.9874267578125
Date: 2025-01-02 Close Price: 242.9874267578125
Close Price: 242.4991455078125
Date: 2025-01-03 Close Price: 242.4991455078125
Close Price: 244.1333465576172
Date: 2025-01-06 Close Price: 244.1333465576172
Close Price: 241.3532257080078
Date: 2025-01-07 Close Price: 241.3532257080078
Close Price: 241.8414764404297
Date: 2025-01-08 Close Price: 241.8414764404297
Close Price: 236.01217651367188
Date: 2025-01-10 Close Price: 236.01217651367188
Close Price: 233.5708465576172
Date: 2025-01-13 Close Price: 233.5708465576172
Close Price: 232.45480346679688
Date: 2025-01-14 Close Price: 232.45480346679688
Close Price: 237.028564453125
Date: 2025-01-15 Close Price: 237.028564453125
Close Price: 227.45257568359375
Date: 2025-01-16 Close Price: 227.45257568359375
Close Price: 229.16647338867188
Date: 2025-01-17 Close Price: 229.16647338867188
Close Price: 221.85244750976562
Date: 2025-01-21 Close Price: 221.85244750976562
Close Price: 223.03822326660156
Date: 2025-0

In [25]:
class MovingAverageCrossStrategy(bt.Strategy):
    params = (("fast_period", 10), ("slow_period", 30),)

    def __init__(self):
        self.fast_ma = bt.indicators.SimpleMovingAverage(self.data.close, period=self.params.fast_period)
        self.slow_ma = bt.indicators.SimpleMovingAverage(self.data.close, period=self.params.slow_period)
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
                # print("BUY")
        else:
            if self.crossover < 0:
                self.sell()
                # print("SELL")

datafeed = bt.feeds.PandasData(dataname=df)
cerebro = bt.Cerebro()

cerebro.adddata(datafeed)

cerebro.addstrategy(MovingAverageCrossStrategy)
cerebro.addobserver(bt.observers.Value)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name="sharpe_ratio", riskfreerate=0)
cerebro.addanalyzer(bt.analyzers.DrawDown, _name="drawdown")
cerebro.addanalyzer(bt.analyzers.Returns, _name="returns")

cerebro.broker.setcash(10000)
cerebro.broker.setcommission(commission=0.01)
print(f"Starting Portfolio Value: {cerebro.broker.getvalue():.2f}")
print(f"Commission: {cerebro.broker.getcommissioninfo(datafeed).getcommission(1, 100):.2f}")
print(f"Starting Cash: {cerebro.broker.getcash():.2f}")
results = cerebro.run()

print(f"Sharpe Ratio: {results[0].analyzers.sharpe_ratio.get_analysis()['sharperatio']}")
print(f"Drawdown: {results[0].analyzers.drawdown.get_analysis()['max']['drawdown']:.2f}%")
print(f"Returns: {results[0].analyzers.returns.get_analysis()['rnorm100']:.2f}%")

print(f"Final Portfolio Value: {cerebro.broker.getvalue():.2f}")

Starting Portfolio Value: 10000.00
Commission: 1.00
Starting Cash: 10000.00
Sharpe Ratio: None
Drawdown: 0.48%
Returns: -0.49%
Final Portfolio Value: 9967.78
